# Ligand-Pocket QGNN: Quantum vs Classical Comparison

Comparison of Quantum and Classical GNN models for ligand-pocket binding classification.

**Architecture:**
- Ligand: Graph Neural Network → Latent Vector
- Pocket: MLP → Latent Vector
- Interaction: Quantum Circuit or Classical MLP → Binding Probability

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import json
from datetime import datetime
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, 
    recall_score, f1_score
)
from tqdm import tqdm
import pandas as pd

from hardware_optimizer import setup_environment
from ligand_pocket_qgnn.data import LigandPocketDataProcessor, LigandPocketDataset, collate_fn
from ligand_pocket_qgnn.model import LigandPocketQGNN

## 1. Auto-Detect Hardware & Configure

In [2]:
# Automatic hardware detection and optimization
hw_info, hw_config = setup_environment()

# Apply hardware-optimized configuration
DEVICE = hw_info['device']
BATCH_SIZE = hw_config['batch_size']
NUM_WORKERS = hw_config['num_workers']
PREFETCH = hw_config['prefetch_factor']
PIN_MEMORY = hw_config['pin_memory']
QUANTUM_DEVICE = hw_config['quantum_device']

# Paths
DATA_DIR = "/Users/priyanshudey/Code/Qunatum copy/othercode/data"
SAVE_DIR = "./ligand_pocket_comparison_results"
os.makedirs(SAVE_DIR, exist_ok=True)

# Reproducibility
SEED = 42069
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(SEED)

# Model hyperparameters
HIDDEN_DIM = 64
N_QUBITS = 6
N_QLAYERS = 2

# Training hyperparameters
EPOCHS = 100
LEARNING_RATE = 0.001
VAL_SPLIT = 0.2
EARLY_STOPPING_PATIENCE = 15

Hardware Optimization: Apple Silicon GPU (MPS)
Config: Batch=256, Workers=11, QDevice=default.qubit


## 2. Load Data

In [3]:
processor = LigandPocketDataProcessor(DATA_DIR, seed=SEED)
processor.load_data(max_samples=None)
interactions = processor.get_dataset()

print(f"Total interactions: {len(interactions)}")

Searching for data in: /Users/priyanshudey/Code/Qunatum copy/othercode/data
Found 15239 protein descriptor files


Loading Data: 100%|██████████| 15239/15239 [00:43<00:00, 349.18it/s]


Generating negative samples (target: 122130)...


Generating Negatives:  35%|███▍      | 42210/122130 [00:00<00:00, 148582.09it/s]

Generated 9997/122130 negatives in batch. Filling remainder...


Generating Negatives: 100%|██████████| 122130/122130 [00:00<00:00, 132841.25it/s]

Loaded 13201 pockets, 122130 ligands
Interactions: 122130 positive, 122130 negative
Total interactions: 244260


## 3. Prepare Datasets

In [4]:
# Import collate_fn from data.py (defined at module level for multiprocessing)
train_ints, val_ints = train_test_split(interactions, test_size=VAL_SPLIT, random_state=SEED)
train_dataset = LigandPocketDataset(processor, train_ints)
val_dataset = LigandPocketDataset(processor, val_ints)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, persistent_workers=NUM_WORKERS > 0,
    prefetch_factor=PREFETCH if NUM_WORKERS > 0 else None
)

sample_ligand = processor.ligands[interactions[0].ligand_id]
sample_pocket = processor.pockets[interactions[0].pocket_id]
ligand_dim = sample_ligand.atom_features.shape[1]
pocket_dim = sample_pocket.to_vector().shape[0]

print(f"Train: {len(train_dataset)} samples ({len(train_loader)} batches)")
print(f"Val: {len(val_dataset)} samples ({len(val_loader)} batches)")
print(f"Ligand dim: {ligand_dim} | Pocket dim: {pocket_dim}")

Train: 195408 samples (764 batches)
Val: 48852 samples (191 batches)
Ligand dim: 10 | Pocket dim: 19


## 4. Training Functions

In [5]:
def train_epoch(model, optimizer, criterion, loader, device):
    model.train()
    total_loss, all_preds, all_labels = 0.0, [], []
    
    for x, edge_idx, batch_vec, pocket, labels in tqdm(loader, desc='Train'):
        x, edge_idx, batch_vec, pocket, labels = (
            x.to(device), edge_idx.to(device), batch_vec.to(device),
            pocket.to(device), labels.to(device)
        )
        
        optimizer.zero_grad(set_to_none=True)
        outputs = model(x, edge_idx, batch_vec, pocket).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item() * len(labels)
        all_preds.extend(outputs.detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    return {
        'loss': total_loss / len(all_labels),
        'accuracy': accuracy_score(all_labels, (np.array(all_preds) >= 0.5).astype(int))
    }


def evaluate(model, criterion, loader, device):
    model.eval()
    total_loss, all_preds, all_labels = 0.0, [], []
    
    with torch.no_grad():
        for x, edge_idx, batch_vec, pocket, labels in tqdm(loader, desc='Val', leave=False):
            x, edge_idx, batch_vec, pocket, labels = (
                x.to(device), edge_idx.to(device), batch_vec.to(device),
                pocket.to(device), labels.to(device)
            )
            
            outputs = model(x, edge_idx, batch_vec, pocket).squeeze()
            loss = criterion(outputs, labels)
            
            total_loss += loss.item() * len(labels)
            all_preds.extend(outputs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_preds_binary = (all_preds >= 0.5).astype(int)
    
    return {
        'loss': total_loss / len(all_labels),
        'accuracy': accuracy_score(all_labels, all_preds_binary),
        'auc': roc_auc_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds_binary, zero_division=0),
        'recall': recall_score(all_labels, all_preds_binary, zero_division=0),
        'f1': f1_score(all_labels, all_preds_binary, zero_division=0)
    }


def train_model(model, train_loader, val_loader, model_name, device):
    print(f"\n{'='*70}\nTraining {model_name.upper()}\n{'='*70}")
    print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    criterion = nn.BCELoss()
    
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 
               'val_auc': [], 'val_precision': [], 'val_recall': [], 'val_f1': []}
    best_val_auc, patience_counter = 0.0, 0
    start_time = datetime.now()
    
    for epoch in range(EPOCHS):
        print(f"\nEpoch {epoch+1}/{EPOCHS}")
        
        train_metrics = train_epoch(model, optimizer, criterion, train_loader, device)
        val_metrics = evaluate(model, criterion, val_loader, device)
        scheduler.step(val_metrics['auc'])
        
        for key in history:
            prefix = 'train_' if key.startswith('train_') else 'val_'
            history[key].append(train_metrics.get(key.replace(prefix, '')) if key.startswith('train_') 
                              else val_metrics.get(key.replace(prefix, '')))
        
        print(f"  Train: Loss={train_metrics['loss']:.4f}, Acc={train_metrics['accuracy']:.4f}")
        print(f"  Val:   Loss={val_metrics['loss']:.4f}, Acc={val_metrics['accuracy']:.4f}, "
              f"AUC={val_metrics['auc']:.4f}, F1={val_metrics['f1']:.4f}")
        
        if val_metrics['auc'] > best_val_auc:
            best_val_auc = val_metrics['auc']
            patience_counter = 0
            torch.save({'model_state_dict': model.state_dict(), 'best_auc': best_val_auc},
                      os.path.join(SAVE_DIR, f"{model_name}_best.pt"))
            print(f"   New best AUC: {best_val_auc:.4f}")
        else:
            patience_counter += 1
            print(f"  Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")
        
        with open(os.path.join(SAVE_DIR, f"{model_name}_history.json"), 'w') as f:
            json.dump(history, f)
        
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    print(f"\nComplete: {(datetime.now() - start_time).total_seconds()/60:.2f} min, Best AUC: {best_val_auc:.4f}")
    return history, best_val_auc

## 5. Train Quantum Model

In [ ]:
quantum_model = LigandPocketQGNN(
    ligand_in_dim=ligand_dim, pocket_in_dim=pocket_dim, hidden_dim=HIDDEN_DIM,
    n_qubits=N_QUBITS, n_qlayers=N_QLAYERS, use_quantum=True, quantum_device=QUANTUM_DEVICE
)

quantum_history, quantum_best_auc = train_model(quantum_model, train_loader, val_loader, "quantum", DEVICE)

✓ Parallel Quantum layer created
  Qubits: 6, Layers: 2, Device: default.qubit
  🚀 Using 12 worker threads for parallel evaluation

Training QUANTUM
Parameters: 10,694

Epoch 1/100


Train:   0%|          | 0/764 [00:00<?, ?it/s]

  ✓ Quantum circuit initialized with 12 OpenMP threads


Train:  14%|█▍        | 106/764 [05:35<36:24,  3.32s/it]

## 6. Train Classical Model

In [ ]:
classical_model = LigandPocketQGNN(
    ligand_in_dim=ligand_dim, pocket_in_dim=pocket_dim, hidden_dim=HIDDEN_DIM,
    n_qubits=N_QUBITS, n_qlayers=N_QLAYERS, use_quantum=False
)

classical_history, classical_best_auc = train_model(classical_model, train_loader, val_loader, "classical", DEVICE)

## 7. Results

In [ ]:
print("\n" + "="*70)
print("FINAL COMPARISON")
print("="*70)
print(f"Quantum:   AUC = {quantum_best_auc:.4f}")
print(f"Classical: AUC = {classical_best_auc:.4f}")
diff = (quantum_best_auc - classical_best_auc) * 100
print(f"Difference: {diff:+.2f}% {'(quantum better)' if diff > 0 else '(classical better)'}")
print("="*70)

# Detailed metrics
q_idx, c_idx = np.argmax(quantum_history['val_auc']), np.argmax(classical_history['val_auc'])
results = pd.DataFrame({
    'Model': ['Quantum', 'Classical'],
    'Epoch': [q_idx + 1, c_idx + 1],
    'Loss': [quantum_history['val_loss'][q_idx], classical_history['val_loss'][c_idx]],
    'Acc': [quantum_history['val_acc'][q_idx], classical_history['val_acc'][c_idx]],
    'AUC': [quantum_best_auc, classical_best_auc],
    'Prec': [quantum_history['val_precision'][q_idx], classical_history['val_precision'][c_idx]],
    'Rec': [quantum_history['val_recall'][q_idx], classical_history['val_recall'][c_idx]],
    'F1': [quantum_history['val_f1'][q_idx], classical_history['val_f1'][c_idx]]
})
print("\n" + results.to_string(index=False))
results.to_csv(os.path.join(SAVE_DIR, 'comparison_results.csv'), index=False)

## 8. Visualization

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
metrics = [('train_loss', 'Train Loss'), ('val_loss', 'Val Loss'), 
           ('train_acc', 'Train Acc'), ('val_acc', 'Val Acc'),
           ('val_auc', 'Val AUC'), ('val_f1', 'Val F1')]

for idx, (key, title) in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    ax.plot(quantum_history[key], label='Quantum', color='blue', alpha=0.7)
    ax.plot(classical_history[key], label='Classical', color='orange', alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'comparison_plots.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f"\nPlots saved to {SAVE_DIR}/comparison_plots.png")